# QLoRA fine-tune — NoteChat dialogue generation

**Task (current data pipeline):** given a clinical note, generate the doctor-patient dialogue conditioned on it. Input/output columns come from `data/processed/{train,val}.parquet`, built by `python -m src.data.build_dataset` (see `docs/data_report.md`).

> **Note on scope:** `src/train/train.py` and `docs/DECISIONS.md` describe a different, CUAD contract-clause-extraction task. The data pipeline actually committed and run in this repo (`configs/data.yaml`, `src/data/build_dataset.py`, the parquet files this notebook loads) builds NoteChat clinical dialogues instead — that mismatch was flagged and this notebook was deliberately written against the data that exists on disk today. If the CUAD pipeline gets built later, this notebook's prompt/target functions (Cell "Task formatting") are the only place that needs to change.

**Before running:**
1. Needs an NVIDIA GPU. This *does* run natively on Windows — `uv sync --extra gpu` installs a working torch/bitsandbytes/unsloth/trl/peft/accelerate stack on win_amd64, no WSL2 required. (Verified on an RTX 3060, 12GB, sm_86.)
2. `uv sync --extra gpu` from the repo root, then select `.venv` as the notebook kernel.
3. **On Windows, torch must come from a CUDA index, not PyPI** — PyPI serves CPU-only torch wheels for win_amd64, which install without error but leave `torch.cuda.is_available() == False` and silently train on CPU. `pyproject.toml` pins torch/torchvision to PyTorch's cu128 index for this reason. Cell 1's assert is what catches a regression here.
4. `MODEL_NAME` below is set to a checkpoint confirmed at build time per `PROJECT_SPEC.md` §7 — change it if you are targeting a different model, and keep `CHAT_TEMPLATE` matching that model's family (e.g. `qwen-2.5`, `llama-3.1`, `mistral`) — see [unsloth's chat_templates](https://github.com/unslothai/unsloth) for supported names.

Hyperparameters come from `configs/train.yaml`, same as the CLI trainer, so the two never drift out of sync.

In [1]:
import json
from pathlib import Path

import polars as pl
import torch
import yaml

assert torch.cuda.is_available(), (
    "No CUDA GPU visible. This notebook needs the GPU training stack "
    "(`uv sync --extra gpu`) on a Linux/NVIDIA machine or WSL2 — see README.md."
)
print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3060


## Config + paths

In [2]:
CONFIG_PATH = Path("../configs/train.yaml")
TRAIN_PARQUET = Path("../data/processed/train.parquet")
VAL_PARQUET = Path("../data/processed/val.parquet")
ADAPTER_OUT = Path("../artifacts/adapters")

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

if not TRAIN_PARQUET.exists():
    raise FileNotFoundError(
        f"{TRAIN_PARQUET} not found — run `python -m src.data.build_dataset` first "
        "(from the repo root, with data/raw/notechat/our_revised_v2.csv present)."
    )

cfg

{'model_name': None,
 'quantization': {'load_in_4bit': True,
  'bnb_4bit_quant_type': 'nf4',
  'bnb_4bit_use_double_quant': True,
  'bnb_4bit_compute_dtype': 'bfloat16'},
 'lora': {'r': 16,
  'lora_alpha': 32,
  'lora_dropout': 0.05,
  'target_modules': ['q_proj',
   'k_proj',
   'v_proj',
   'o_proj',
   'gate_proj',
   'up_proj',
   'down_proj']},
 'max_seq_len': 4096,
 'per_device_batch_size': 1,
 'gradient_accumulation_steps': 16,
 'gradient_checkpointing': True,
 'optim': 'paged_adamw_8bit',
 'learning_rate': 0.0002,
 'lr_scheduler_type': 'cosine',
 'warmup_ratio': 0.03,
 'num_train_epochs': 2,
 'packing': True,
 'seed': 42,
 'logging_steps': 50,
 'save_steps': 50,
 'save_total_limit': 3}

In [3]:
# --- set these before running ---
# Confirmed at build time per PROJECT_SPEC.md §7 (#1/#4) rather than assumed:
# unsloth's pre-quantized 4-bit Qwen2.5-3B-Instruct. Matches the "~4B model on
# a 12GB card" sizing configs/train.yaml targets, and was checked against the
# actual training GPU (RTX 3060, 12GB, sm_86 — Ampere, so the config's
# bfloat16 compute dtype is supported). Pre-quantized, so the download is
# ~2GB rather than ~6GB and the load-time spike of on-the-fly quantization
# is avoided.
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
CHAT_TEMPLATE = "qwen-2.5"  # must match MODEL_NAME's family

assert MODEL_NAME, "Set MODEL_NAME above before running the rest of the notebook."

## Task formatting

Input = clinical note, target = the synthetic doctor/patient dialogue conditioned on it (`docs/data_report.md`'s "Ground-truth caveat": the dialogue is itself LLM-generated, not human-authored).

In [4]:
SYSTEM_PROMPT = (
    "You are a clinical documentation assistant. Given a clinical note, generate "
    "a realistic doctor-patient dialogue consistent with the note. Output only the "
    "dialogue, formatted as alternating `Doctor:`/`Patient:` turns — no other text."
)


def build_user_prompt(row: dict) -> str:
    return f"Clinical note:\n{row['clinical_note']}"


def build_target(row: dict) -> str:
    return row["conversation"]


def to_chat_text(row: dict, tokenizer) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
        {"role": "assistant", "content": build_target(row)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_split(path: Path, tokenizer):
    from datasets import Dataset

    df = pl.read_parquet(path)
    texts = [to_chat_text(row, tokenizer) for row in df.iter_rows(named=True)]
    return Dataset.from_dict({"text": texts})

## Load model + apply QLoRA

`unsloth` must be imported before `trl`/`transformers`/`peft` so its patches apply.

In [5]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

compute_dtype = cfg["quantization"]["bnb_4bit_compute_dtype"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=cfg["max_seq_len"],
    dtype=None,  # auto: bfloat16 if supported else float16
    load_in_4bit=cfg["quantization"]["load_in_4bit"],
    random_state=cfg["seed"],
)
tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)

model = FastLanguageModel.get_peft_model(
    model,
    r=cfg["lora"]["r"],
    target_modules=cfg["lora"]["target_modules"],
    lora_alpha=cfg["lora"]["lora_alpha"],
    lora_dropout=cfg["lora"]["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth" if cfg["gradient_checkpointing"] else False,
    random_state=cfg["seed"],
    max_seq_length=cfg["max_seq_len"],
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\bmsip\company-finetune-eval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0824 19:05:26.650000 8 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 12.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 290.50it/s]
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.8.19 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Build datasets

In [6]:
train_ds = load_split(TRAIN_PARQUET, tokenizer)
val_ds = load_split(VAL_PARQUET, tokenizer) if VAL_PARQUET.exists() else None

print(f"train: {len(train_ds)} examples")
if val_ds is not None:
    print(f"val: {len(val_ds)} examples")
print("---\n" + train_ds[0]["text"][:1500])

train: 8000 examples
val: 1000 examples
---
<|im_start|>system
You are a clinical documentation assistant. Given a clinical note, generate a realistic doctor-patient dialogue consistent with the note. Output only the dialogue, formatted as alternating `Doctor:`/`Patient:` turns — no other text.<|im_end|>
<|im_start|>user
Clinical note:
A 52-year-old woman with multiple comorbidities, including obesity and chronic use of prednisone presumably for pulmonary fibrosis, originally presented to an urgent care center two days prior to presenting to our academic hospital and was prescribed polymyxin for presumed conjunctivitis. The patient then presented to our community campus emergency department (ED) because she felt that her “head is swollen and feels like her throat [is] starting to swell.” She believed she was having an allergic reaction; after using her EpiPen® without resolution, she came to the ED to be treated. On initial exam her vital signs were blood pressure 160/90 millimeters of

## Train

Loss is logged and a checkpoint written every `logging_steps`/`save_steps` optimizer steps (both 50 in `configs/train.yaml`). One optimizer step is `per_device_batch_size * gradient_accumulation_steps` examples, so with the shipped config 50 steps ≈ 800 examples. Checkpoints land in `run_dir/checkpoint-<step>/`; only the 3 newest are kept (`save_total_limit`), so a long run can't fill the disk. Resume an interrupted run with `trainer.train(resume_from_checkpoint=True)`.

In [7]:
from trl import SFTConfig, SFTTrainer

run_dir = ADAPTER_OUT / MODEL_NAME.replace("/", "__")
run_dir.mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=str(run_dir),
    per_device_train_batch_size=cfg["per_device_batch_size"],
    gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
    learning_rate=cfg["learning_rate"],
    lr_scheduler_type=cfg["lr_scheduler_type"],
    warmup_ratio=cfg["warmup_ratio"],
    num_train_epochs=cfg["num_train_epochs"],
    optim=cfg["optim"],
    seed=cfg["seed"],
    max_length=cfg["max_seq_len"],
    packing=cfg["packing"],
    dataset_text_field="text",
    bf16=(compute_dtype == "bfloat16"),
    fp16=(compute_dtype != "bfloat16"),
    gradient_checkpointing=cfg["gradient_checkpointing"],
    logging_steps=cfg["logging_steps"],
    save_strategy="steps",
    save_steps=cfg["save_steps"],
    save_total_limit=cfg["save_total_limit"],
    eval_strategy="epoch" if val_ds is not None else "no",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print(f"Training {MODEL_NAME} on {len(train_ds)} notes -> {run_dir}")
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Unsloth: Packing eval dataset: 100%|██████████| 1000/1000 [00:00<00:00, 83019.36 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Training unsloth/Qwen2.5-3B-Instruct-bnb-4bit on 8000 notes -> ..\artifacts\adapters\unsloth__Qwen2.5-3B-Instruct-bnb-4bit


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,986 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.081330,1.063326
2,1.045007,1.051183


Unsloth: Restored added_tokens_decoder metadata in ..\artifacts\adapters\unsloth__Qwen2.5-3B-Instruct-bnb-4bit\checkpoint-50\tokenizer_config.json.
c:\Users\bmsip\company-finetune-eval\.venv\Lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error timed out - silently ignoring the lookup for the file config.json in unsloth/Qwen2.5-3B-Instruct-bnb-4bit.
  warnings.warn(
c:\Users\bmsip\company-finetune-eval\.venv\Lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in unsloth/Qwen2.5-3B-Instruct-bnb-4bit - will assume that the vocabulary was not modified.
  warnings.warn(
Unsloth: Restored added_tokens_decoder metadata in ..\artifacts\adapters\unsloth__Qwen2.5-3B-Instruct-bnb-4bit\checkpoint-100\tokenizer_config.json.
c:\Users\bmsip\company-finetune-eval\.venv\Lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error timed out - silently ign

TrainOutput(global_step=250, training_loss=1.0929667358398438, metrics={'train_runtime': 18802.7864, 'train_samples_per_second': 0.211, 'train_steps_per_second': 0.013, 'total_flos': 2.6719253030417203e+17, 'train_loss': 1.0929667358398438, 'epoch': 2.0})

## Save adapter

In [ ]:
final_dir = run_dir / "final_adapter"
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Saved adapter to {final_dir}")

## Sanity-check generation

In [ ]:
FastLanguageModel.for_inference(model)

sample = pl.read_parquet(VAL_PARQUET if VAL_PARQUET.exists() else TRAIN_PARQUET).row(0, named=True)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": build_user_prompt(sample)},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

output = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True))